# Data Preprocessing

This notebook prepares the raw dataset for machine learning model training.

The preprocessing steps are based on the findings and decisions made during the exploratory data analysis.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [3]:
df = pd.read_csv('/content/signal_metrics.csv')
# df = pd.read_csv('../data/raw/signal_metrics.csv')

In [ ]:
df.head()

## 2. Remove Unnecessary Features

Based on the EDA findings, several features were removed before model training.

- `Timestamp` was removed because no clear temporal pattern was observed.
- `Signal Quality (%)` was removed because it is a constant feature with zero variance.
- `BB60C Measurement (dBm)`, `srsRAN Measurement (dBm)`, and `BladeRFxA9 Measurement (dBm)` were removed because they provide highly redundant information.

The target variable, `Data Throughput (Mbps)`, is retained for the prediction task.

In [13]:
df_processed = df.copy()

In [14]:
columns_to_drop = [
    'Timestamp',
    'Signal Quality (%)',
    'BB60C Measurement (dBm)',
    'srsRAN Measurement (dBm)',
    'BladeRFxA9 Measurement (dBm)'
]

df_processed = df_processed.drop(columns = columns_to_drop)

df_processed.head()

,Locality,Latitude,Longitude,Signal Strength (dBm),Data Throughput (Mbps),Latency (ms),Network Type
0,Anisabad,25.599109,85.137355,-84.274113,1.863890,129.122914,3G
1,Fraser Road,25.433286,85.070053,-97.653121,5.132296,54.883606,4G
2,Boring Canal Road,25.498809,85.211371,-87.046134,1.176985,119.598286,LTE
3,Danapur,25.735138,85.208400,-94.143159,68.596932,46.598387,5G
4,Phulwari Sharif,25.538556,85.159860,-94.564765,38.292038,30.342828,5G


In [15]:
df_processed['Network Type'] = df_processed['Network Type'].replace({
    'LTE': '4G'
})

In [17]:
df_processed['Network Type'].value_counts()

,count
Network Type,
4G,8443
3G,4208
5G,4178


In [16]:
df_processed.columns

Index(['Locality', 'Latitude', 'Longitude', 'Signal Strength (dBm)',
       'Data Throughput (Mbps)', 'Latency (ms)', 'Network Type'],
      dtype='object')

## 3. Separate Features and Target

The target variable is **Data Throughput (Mbps)**, which is the continuous value we want to predict.

The remaining columns are used as input features for the machine learning models.

In [18]:
X = df_processed.drop(columns='Data Throughput (Mbps)')
y = df_processed['Data Throughput (Mbps)']

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')

X shape : (16829, 6)
y shape : (16829,)


## 4. Train/Test Split

The dataset was divided into training and testing sets using an 80/20 split.

The training set will be used to train the machine learning models, while the testing set will be reserved for evaluating their performance on unseen data.

A fixed random state is used to ensure reproducibility.

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state = 42
)

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')
print(f'y_train shape {y_train.shape}')
print(f'y_test shape {y_test.shape}')

X_train shape (13463, 6)
X_test shape (3366, 6)
y_train shape (13463,)
y_test shape (3366,)


## 5. Identify Numerical and Categorical Features

The input features were divided into numerical and categorical variables because they require different preprocessing techniques.

Numerical features can be processed using numerical transformations such as scaling, while categorical features need to be encoded into numerical representations before being used by machine learning models.

In [20]:
numerical_features = X_train.select_dtypes(include = 'number').columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

print("Numerical features\n" , numerical_features)
print("\nCategorical features\n", categorical_features)

Numerical features
 ['Latitude', 'Longitude', 'Signal Strength (dBm)', 'Latency (ms)']

Categorical features
 ['Locality', 'Network Type']


## 6. Build the Preprocessing Pipeline

A preprocessing pipeline was created to handle numerical and categorical features separately.

- Numerical features are standardized using `StandardScaler`.
- Categorical features are converted into numerical representations using `OneHotEncoder`.
- `ColumnTransformer` is used to apply the appropriate transformation to each group of features.

The preprocessing transformations will be fitted using the training data only to prevent data leakage.

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'),categorical_features)
    ]
)

preprocessor.fit_transform(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 80778 stored elements and shape (13463, 27)>

## 7. Fit and Transform the Training Data

The preprocessing pipeline was fitted and applied to the training data.

During the fitting process:

- `StandardScaler` learned the mean and standard deviation of the numerical features from the training set.
- `OneHotEncoder` learned the categorical levels present in the training set.

The fitted transformations were then applied to the training data.

In [22]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Original X_train shape : ", X_train.shape)
print(f'Processed X_train shape : {X_train_processed.shape}')

Original X_train shape :  (13463, 6)
Processed X_train shape : (13463, 27)


In [23]:
X_test_processed = preprocessor.transform(X_test)


print("Original X_test shape:", X_test.shape)
print("Processed X_test shape:", X_test_processed.shape)

Original X_test shape: (3366, 6)
Processed X_test shape: (3366, 27)


## 8. Inspect the Processed Data

After applying the preprocessing pipeline, we inspect the resulting feature representation and verify that the transformations were applied correctly.

In [24]:
print("Processed data type:", type(X_train_processed))
print("Number of features:", X_train_processed.shape[1])

Processed data type: <class 'scipy.sparse._csr.csr_matrix'>
Number of features: 27


In [25]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__Latitude' 'num__Longitude' 'num__Signal Strength (dBm)'
 'num__Latency (ms)' 'cat__Locality_Anandpuri' 'cat__Locality_Anisabad'
 'cat__Locality_Ashok Rajpath' 'cat__Locality_Bailey Road'
 'cat__Locality_Bankipore' 'cat__Locality_Boring Canal Road'
 'cat__Locality_Boring Road' 'cat__Locality_Danapur'
 'cat__Locality_Exhibition Road' 'cat__Locality_Fraser Road'
 'cat__Locality_Gandhi Maidan' 'cat__Locality_Gardanibagh'
 'cat__Locality_Kankarbagh' 'cat__Locality_Kidwaipuri'
 'cat__Locality_Kumhrar' 'cat__Locality_Pataliputra'
 'cat__Locality_Patliputra Colony' 'cat__Locality_Phulwari Sharif'
 'cat__Locality_Rajendra Nagar' 'cat__Locality_S.K. Puri'
 'cat__Network Type_3G' 'cat__Network Type_4G' 'cat__Network Type_5G']


## 9. Save the Train/Test Sets

The training and testing sets are saved separately in the Colab working directory.

Saving the datasets separately allows the same train/test split to be reused across different modeling notebooks while maintaining consistent and reproducible model evaluation.

The datasets are saved before scaling and categorical encoding. These preprocessing steps will be applied later using the training data only to prevent data leakage.


In [26]:
os.makedirs('/content/processed', exist_ok=True)
X_train.to_csv('/content/processed/X_train.csv', index=False)
X_test.to_csv('/content/processed/X_test.csv', index=False)

y_train.to_csv('/content/processed/y_train.csv', index=False)
y_test.to_csv('/content/processed/y_test.csv', index=False)
df_processed.to_csv('/content/processed/processed_data.csv', index = False)

print("Train/test sets saved successfully.")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Train/test sets saved successfully.
X_train: (13463, 6)
X_test: (3366, 6)
y_train: (13463,)
y_test: (3366,)


In [27]:
X_train.head()

,Locality,Latitude,Longitude,Signal Strength (dBm),Latency (ms),Network Type
12573,Pataliputra,25.605539,85.285651,-87.611830,164.101054,4G
5139,Kidwaipuri,25.491973,85.086669,-90.089704,22.851745,5G
5900,Danapur,25.684972,84.989785,-90.175391,139.812003,4G
3328,Kankarbagh,25.569868,85.138714,-93.384826,113.904981,4G
8457,Phulwari Sharif,25.626811,85.136823,-95.888704,71.291449,4G
